In [1]:
import pandas as pd
import numpy as np

path = r"C:\Users\Arifi\Desktop\FORECASTING OBAT\data\processed\Concat.csv"
label = r"C:\Users\Arifi\Desktop\FORECASTING OBAT\data\final\label.xlsx"

df = pd.read_csv(path, low_memory=False)
label = pd.read_excel(label)

df['Tanggal Transaksi'] = pd.to_datetime(df['Tanggal Transaksi'])
print("Raw shape:", df.shape)

Raw shape: (27346, 14)


In [2]:
label['cat_norm'] = label['Draf_Kategori'].str.strip().str.lower()
obat_skus = set(label[label['cat_norm'] == 'obat']['SKU'].dropna().unique())

df_obat = df[df['Kode Produk'].isin(obat_skus)].copy()
print(f"SKU obat di master data: {len(obat_skus)}")
print(f"Baris transaksi obat: {len(df_obat)}")
print(f"SKU obat bertransaksi: {df_obat['Kode Produk'].nunique()}")

SKU obat di master data: 937
Baris transaksi obat: 18404
SKU obat bertransaksi: 847


In [3]:
df_obat['week'] = df_obat['Tanggal Transaksi'].dt.to_period('W-MON')
weekly = df_obat.groupby(['Kode Produk', 'week'])['Jumlah'].sum().reset_index()

print("Shape setelah agregasi mingguan:", weekly.shape)
print(weekly.head())

Shape setelah agregasi mingguan: (9563, 3)
    Kode Produk                   week  Jumlah
0    AK-ALTLATL  2026-01-27/2026-02-02       1
1    AK-ALTLATL  2026-02-03/2026-02-09       1
2  AK-ONEKOTP3K  2026-03-24/2026-03-30       1
3        ASF-01  2025-04-01/2025-04-07       3
4        ASF-01  2025-04-22/2025-04-28       3


In [4]:
all_weeks = pd.period_range(
    start=df_obat['Tanggal Transaksi'].min(),
    end=df_obat['Tanggal Transaksi'].max(),
    freq='W-MON'
)
all_skus = df_obat['Kode Produk'].unique()

idx = pd.MultiIndex.from_product([all_skus, all_weeks], names=['Kode Produk', 'week'])
full_matrix = pd.DataFrame(index=idx).reset_index()
full_matrix = full_matrix.merge(weekly, on=['Kode Produk', 'week'], how='left')
full_matrix['Jumlah'] = full_matrix['Jumlah'].fillna(0).astype(int)

print(f"Total minggu: {len(all_weeks)}")
print(f"Matriks penuh: {full_matrix.shape}")

Total minggu: 57
Matriks penuh: (48279, 3)


In [5]:
weeks_active = full_matrix[full_matrix['Jumlah'] > 0].groupby('Kode Produk')['week'].nunique()
fast_moving_skus = weeks_active[weeks_active >= 17].index.tolist()

fm_matrix = full_matrix[full_matrix['Kode Produk'].isin(fast_moving_skus)].copy()

print(f"Fast-moving SKU: {len(fast_moving_skus)}")
print(f"Slow-moving SKU: {len(all_skus) - len(fast_moving_skus)}")
print(f"Matriks fast-moving: {fm_matrix.shape}")

# ── Metadata klasifikasi untuk transparansi DSS ──
THRESHOLD = 17
TOTAL_MINGGU = len(all_weeks)

sku_class = pd.DataFrame({'Kode Produk': all_skus})
sku_class['Minggu_Aktif'] = sku_class['Kode Produk'].map(weeks_active).fillna(0).astype(int)
sku_class['Total_Minggu'] = TOTAL_MINGGU
sku_class['Threshold']    = THRESHOLD
sku_class['Kategori']     = sku_class['Minggu_Aktif'].apply(
    lambda x: 'Fast-Moving' if x >= THRESHOLD else 'Slow-Moving'
)
sku_to_nama = dict(zip(label['SKU'], label['Nama']))
sku_class['Nama Produk'] = sku_class['Kode Produk'].map(sku_to_nama).fillna('-')

print(f"Klasifikasi: {len(sku_class)} SKU")
print(f"  Fast-Moving: {(sku_class['Kategori']=='Fast-Moving').sum()}")
print(f"  Slow-Moving: {(sku_class['Kategori']=='Slow-Moving').sum()}")


Fast-moving SKU: 195
Slow-moving SKU: 652
Matriks fast-moving: (11115, 3)
Klasifikasi: 847 SKU
  Fast-Moving: 195
  Slow-Moving: 652


In [6]:
fm_matrix = fm_matrix.sort_values(['Kode Produk', 'week']).reset_index(drop=True)

# Rata historis per SKU (dihitung sebelum lag, dari seluruh histori)
fm_matrix['Rata_Historis_SKU'] = (
    fm_matrix.groupby('Kode Produk')['Jumlah']
    .expanding().mean().shift(1)
    .reset_index(level=0, drop=True)
)

# Lag features
fm_matrix['Lag_1'] = fm_matrix.groupby('Kode Produk')['Jumlah'].shift(1)
fm_matrix['Lag_2'] = fm_matrix.groupby('Kode Produk')['Jumlah'].shift(2)
fm_matrix['Lag_3'] = fm_matrix.groupby('Kode Produk')['Jumlah'].shift(3)
fm_matrix['Lag_4'] = fm_matrix.groupby('Kode Produk')['Jumlah'].shift(4)

# Rolling mean 4 minggu
fm_matrix['Rolling_Mean_4'] = fm_matrix.groupby('Kode Produk')['Jumlah'].transform(
    lambda x: x.shift(1).rolling(4).mean()
)
# Rolling mean 2 dan 8 minggu
fm_matrix['Rolling_Mean_2'] = fm_matrix.groupby('Kode Produk')['Jumlah'].transform(
    lambda x: x.shift(1).rolling(2).mean()
)
# Rolling std 4 minggu (volatilitas)
fm_matrix['Rolling_Std_4'] = fm_matrix.groupby('Kode Produk')['Jumlah'].transform(
    lambda x: x.shift(1).rolling(4).std()
)

# Temporal features
fm_matrix['Tanggal'] = fm_matrix['week'].dt.start_time
fm_matrix['Bulan'] = fm_matrix['Tanggal'].dt.month
fm_matrix['Pekan_Ke'] = fm_matrix['Tanggal'].dt.isocalendar().week.astype(int)
# Fitur Ramadan
ramadan_start = pd.Timestamp('2026-03-10')
ramadan_end   = pd.Timestamp('2026-04-08')
fm_matrix['Is_Ramadan'] = fm_matrix['Tanggal'].between(ramadan_start, ramadan_end).astype(int)

print("Shape sebelum drop NaN:", fm_matrix.shape)

Shape sebelum drop NaN: (11115, 15)


In [7]:
feature_cols = ['Jumlah', 'Lag_1', 'Lag_2', 'Lag_3', 'Lag_4', 'Rolling_Mean_2',
                'Rolling_Mean_4', 'Rolling_Std_4','Bulan', 
                'Pekan_Ke', 'Rata_Historis_SKU', 'Is_Ramadan']

fm_matrix = fm_matrix.dropna(subset=feature_cols).reset_index(drop=True)

dataset = fm_matrix[['Kode Produk', 'Tanggal'] + feature_cols].copy()

print("Shape dataset final:", dataset.shape)
print(dataset.head())

Shape dataset final: (10335, 14)
  Kode Produk    Tanggal  Jumlah  Lag_1  Lag_2  Lag_3  Lag_4  Rolling_Mean_2  \
0      ASF-01 2025-04-29       1    3.0    0.0    0.0    3.0             1.5   
1      ASF-01 2025-05-06       0    1.0    3.0    0.0    0.0             2.0   
2      ASF-01 2025-05-13       0    0.0    1.0    3.0    0.0             0.5   
3      ASF-01 2025-05-20       0    0.0    0.0    1.0    3.0             0.0   
4      ASF-01 2025-05-27       0    0.0    0.0    0.0    1.0             0.0   

   Rolling_Mean_4  Rolling_Std_4  Bulan  Pekan_Ke  Rata_Historis_SKU  \
0            1.50       1.732051      4        18           1.500000   
1            1.00       1.414214      5        19           1.400000   
2            1.00       1.414214      5        20           1.166667   
3            1.00       1.414214      5        21           1.000000   
4            0.25       0.500000      5        22           0.875000   

   Is_Ramadan  
0           0  
1           0  
2    

In [8]:
cols_to_winsorize = ['Jumlah', 'Lag_1', 'Lag_2', 'Lag_3', 'Lag_4', 'Rolling_Mean_2', 
                    'Rolling_Mean_4', 'Rolling_Std_4']
winsor_bounds = {}
total_capped = 0

for col in cols_to_winsorize:
    dataset[col] = dataset[col].astype(float)
    bounds = {}

    for sku, group in dataset.groupby('Kode Produk'):
        Q1 = group[col].quantile(0.25)
        Q3 = group[col].quantile(0.75)
        IQR = Q3 - Q1
        UB = Q3 + 1.5 * IQR
        bounds[sku] = UB
        mask = (dataset['Kode Produk'] == sku) & (dataset[col] > UB)
        total_capped += mask.sum()
        dataset.loc[mask, col] = UB
    winsor_bounds[col] = bounds

print(f"Total titik data di-cap (winsorized): {total_capped}")
print(f"Persentase dari total sel: {total_capped / (len(dataset) * len(cols_to_winsorize)) * 100:.2f}%")

Total titik data di-cap (winsorized): 3008
Persentase dari total sel: 3.64%


In [9]:
dataset.to_csv(r'C:\Users\Arifi\Desktop\FORECASTING OBAT\data\final\dataset_xgboost_ready.csv', index=False)

import pickle
with open('winsor_bounds.pkl', 'wb') as f:
    pickle.dump(winsor_bounds, f)

fast_moving_list = pd.DataFrame({'Kode Produk': fast_moving_skus})
fast_moving_list.to_csv(r'C:\Users\Arifi\Desktop\FORECASTING OBAT\data\final\fast_moving_skus.csv', index=False)

print("Tersimpan:")
print(f"  dataset_xgboost_ready.csv — {dataset.shape}")
print(f"  winsor_bounds.pkl — batas winsorization per SKU per kolom")
print(f"  fast_moving_skus.csv — daftar {len(fast_moving_skus)} SKU fast-moving")

# Save klasifikasi SKU
sku_class.to_csv(r'C:\Users\Arifi\Desktop\FORECASTING OBAT\data\final\sku_classification.csv', index=False)
print(f"  sku_classification.csv — {len(sku_class)} SKU klasifikasi")


Tersimpan:
  dataset_xgboost_ready.csv — (10335, 14)
  winsor_bounds.pkl — batas winsorization per SKU per kolom
  fast_moving_skus.csv — daftar 195 SKU fast-moving
  sku_classification.csv — 847 SKU klasifikasi
